# Predicción con ResNet4 manual GELU

Este notebook carga el modelo guardado en `.pth`, hace la predicción sobre `X_test.npz`, aplica `softmax` para obtener probabilidades y guarda el resultado como:

```python
np.savez("Y_pred.npz", Y=Y)
```

La salida esperada es una matriz `Y` de tamaño:

```python
(número de imágenes de X_test, 40)
```

Para el archivo proporcionado, `X_test.npz` contiene `X` con forma `(160, 5600)`, por lo que se espera `Y.shape == (160, 40)`.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path


In [2]:
# =========================
# CONFIGURACIÓN
# =========================

X_TEST_PATH = "X_test.npz"

# Pon aquí el nombre exacto de tu .pth.
# Si lo dejas como None, el notebook buscará automáticamente archivos .pth en la carpeta actual.
PTH_PATH = None
# Ejemplo:
# PTH_PATH = "resnet18_4_0.pth"

OUT_PATH = "Y_pred.npz"

N_CLASSES = 40
BATCH_SIZE = 64

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


## Cargar `X_test.npz`

In [3]:
def cargar_npz(ruta):
    data = np.load(ruta)

    if isinstance(data, np.lib.npyio.NpzFile):
        if len(data.files) == 0:
            raise ValueError(f"El archivo {ruta} no contiene arrays.")

        # En tu X_test.npz la clave es 'X'
        if "X" in data.files:
            return data["X"]

        # Si no existe 'X', usa el primer array del npz
        return data[data.files[0]]

    return data


X_test = cargar_npz(X_TEST_PATH)

print("X_test shape:", X_test.shape)
print("X_test dtype:", X_test.dtype)

assert X_test.ndim == 2, "X_test debe tener forma (n_imágenes, 5600)."
assert X_test.shape[1] == 5600, "Cada imagen debe venir aplanada con 5600 píxeles = 80*70."

# Importante: en el entrenamiento se usaban los valores originales pasados a float32,
# no se normalizaban dividiendo entre 255.
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

test_loader = DataLoader(
    TensorDataset(X_test_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False
)


X_test shape: (160, 5600)
X_test dtype: uint8


## Definición de la red `ResNet4`

In [4]:

class ResNet4(nn.Module): 
    def __init__(self, 
            dim_out,
            activation, 
            apply_bn,
            drop_prob,
            link_function, 
            loss_function
        ):
        super(ResNet4, self).__init__()
        
        self.activation = activation
        self.apply_bn = apply_bn
        self.drop = nn.Dropout2d(drop_prob)
        self.link = link_function
        self.loss = loss_function

        ## ===================== ##
        # Capa de Entrada (1 Canal -> 64 Canales)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        ## ===================== ##
        # RESNET BLOCK 1 (64 -> 64)
        self.b1_conv1 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.b1_bn1 = nn.BatchNorm2d(64)
        self.b1_conv2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.b1_bn2 = nn.BatchNorm2d(64)

        ## ===================== ##
        # RESNET BLOCK 2 (64 -> 128)
        self.b2_conv1 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False)
        self.b2_bn1 = nn.BatchNorm2d(128)
        self.b2_conv2 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.b2_bn2 = nn.BatchNorm2d(128)
        
        self.b2_shortcut_conv = nn.Conv2d(64, 128, kernel_size=1, stride=2, padding=0, bias=False)
        self.b2_shortcut_bn = nn.BatchNorm2d(128)

        ## ===================== ##
        # RESNET BLOCK 3 (128 -> 128)
        self.b3_conv1 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.b3_bn1 = nn.BatchNorm2d(128)
        self.b3_conv2 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.b3_bn2 = nn.BatchNorm2d(128)

        ## ===================== ##
        # RESNET BLOCK 4 (128 -> 256)
        self.b4_conv1 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False)
        self.b4_bn1 = nn.BatchNorm2d(256)
        self.b4_conv2 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=False)
        self.b4_bn2 = nn.BatchNorm2d(256)
        
        self.b4_shortcut_conv = nn.Conv2d(128, 256, kernel_size=1, stride=2, padding=0, bias=False)
        self.b4_shortcut_bn = nn.BatchNorm2d(256)
        
        ## ===================== ##
        self.average_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, dim_out)

    def operator(self, x):
        # Reconstruir la imagen 2D si viene aplanada
        if x.dim() == 2 and x.size(1) == 5600:
            x = x.view(-1, 1, 80, 70)

        feature_maps = {}

        x = self.conv1(x)
        if self.apply_bn:
            x = self.bn1(x)
        x = self.activation(x)
        feature_maps['layer1'] = x.clone()

        ## Bloque 1
        x_block = self.b1_conv1(x)
        if self.apply_bn: x_block = self.b1_bn1(x_block)
        x_block = self.b1_conv2(self.drop(self.activation(x_block)))
        if self.apply_bn: x_block = self.b1_bn2(x_block)
        x = self.drop(self.activation(x + x_block))
        feature_maps['layer2'] = x.clone()

        ## Bloque 2
        x_block = self.b2_conv1(x)
        if self.apply_bn: x_block = self.b2_bn1(x_block)
        x_block = self.b2_conv2(self.drop(self.activation(x_block)))
        if self.apply_bn: x_block = self.b2_bn2(x_block)
        input_readapted = self.b2_shortcut_conv(x)
        if self.apply_bn: input_readapted = self.b2_shortcut_bn(input_readapted) 
        x = self.drop(self.activation(input_readapted + x_block)) 
        feature_maps['layer3'] = x.clone()

        ## Bloque 3
        x_block = self.b3_conv1(x)
        if self.apply_bn: x_block = self.b3_bn1(x_block)
        x_block = self.b3_conv2(self.drop(self.activation(x_block)))
        if self.apply_bn: x_block = self.b3_bn2(x_block)
        x = self.drop(self.activation(x + x_block))
        feature_maps['layer4'] = x.clone()

        ## Bloque 4
        x_block = self.b4_conv1(x)
        if self.apply_bn: x_block = self.b4_bn1(x_block)
        x_block = self.b4_conv2(self.drop(self.activation(x_block)))
        if self.apply_bn: x_block = self.b4_bn2(x_block)
        input_readapted = self.b4_shortcut_conv(x)
        if self.apply_bn: input_readapted = self.b4_shortcut_bn(input_readapted) 
        x = self.drop(self.activation(input_readapted + x_block)) 
        feature_maps['layer5'] = x.clone()

        x = self.average_pooling(x)
        feature_maps['layer_pooling'] = x.clone()
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

    def forward_train(self, x, apply_link=False):
        self.train()
        x = self.operator(x)
        if apply_link: x = self.link(x)
        return x

    def forward_eval(self, x, apply_link=True): 
        self.eval()
        x = self.operator(x)
        if apply_link: x = self.link(x)
        return x

    def forward(self, x):
        return self.operator(x)

    def compute_loss(self, t, y):
        return self.loss(y, t)

## Instanciar el modelo manual con GELU

In [5]:
# =========================
# MODELO MANUAL GUARDADO
# =========================
# Configuración del modelo del mejor guardado manual:
# - activación GELU
# - batch norm activado
# - salida de 40 clases
#
# drop_prob no afecta durante inferencia porque el modelo estará en eval(),
# pero se mantiene el valor del modelo manual.

model = ResNet4(
    dim_out=N_CLASSES,
    activation=F.gelu,
    apply_bn=True,
    drop_prob=0.11019378538869098,
    link_function=nn.Softmax(dim=1),
    loss_function=nn.CrossEntropyLoss()
).to(device)

model


ResNet4(
  (drop): Dropout2d(p=0.11019378538869098, inplace=False)
  (link): Softmax(dim=1)
  (loss): CrossEntropyLoss()
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (b1_conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (b1_bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (b1_conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (b1_bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (b2_conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (b2_bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (b2_conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (b2_bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, 

## Cargar pesos `.pth`

In [6]:
def resolver_pth(pth_path=None):
    if pth_path is not None:
        p = Path(pth_path)
        if not p.exists():
            raise FileNotFoundError(f"No se encuentra el archivo .pth indicado: {p}")
        return p

    pths = sorted(Path(".").glob("*.pth"))

    if len(pths) == 0:
        raise FileNotFoundError(
            "No se ha encontrado ningún archivo .pth en la carpeta actual. "
            "Sube/copia el .pth junto al notebook o pon su nombre en PTH_PATH."
        )

    if len(pths) > 1:
        print("Se han encontrado varios .pth:")
        for p in pths:
            print(" -", p)
        raise ValueError(
            "Hay varios .pth. Indica cuál usar cambiando PTH_PATH, por ejemplo: "
            'PTH_PATH = "nombre_del_modelo.pth"'
        )

    return pths[0]


def extraer_state_dict(checkpoint):
    # Caso normal: torch.save(model.state_dict(), "modelo.pth")
    if isinstance(checkpoint, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]

        # Si parece ya un state_dict
        if all(hasattr(v, "shape") for v in checkpoint.values()):
            return checkpoint

    return checkpoint


pth_file = resolver_pth(PTH_PATH)
print("Cargando modelo desde:", pth_file)

checkpoint = torch.load(pth_file, map_location=device)
state_dict = extraer_state_dict(checkpoint)

# Por si el modelo se guardó con DataParallel y las claves tienen prefijo "module."
state_dict = {
    k.replace("module.", "", 1): v
    for k, v in state_dict.items()
}

model.load_state_dict(state_dict, strict=True)
model.eval()

print("Modelo cargado correctamente.")


Cargando modelo desde: Best_model.pth
Modelo cargado correctamente.


## Predecir probabilidades y guardar `Y_pred.npz`

In [7]:
preds = []

model.eval()
with torch.no_grad():
    for (x_batch,) in test_loader:
        x_batch = x_batch.to(device)

        # Sacamos logits sin aplicar link_function para aplicar softmax explícitamente aquí.
        logits = model.forward_eval(x_batch, apply_link=False)
        probs = torch.softmax(logits, dim=1)

        preds.append(probs.cpu().numpy())

Y = np.concatenate(preds, axis=0).astype(np.float32)

print("Y shape:", Y.shape)
print("Y dtype:", Y.dtype)
print("Suma primera fila:", Y[0].sum())
print("Mínimo:", Y.min(), "Máximo:", Y.max())

assert Y.shape == (X_test.shape[0], N_CLASSES), f"Shape incorrecta: {Y.shape}"
assert np.allclose(Y.sum(axis=1), 1.0, atol=1e-5), "Las filas de Y no suman 1. Revisa el softmax."

np.savez(OUT_PATH, Y=Y)

print(f"Guardado {OUT_PATH} con clave 'Y' y shape {Y.shape}")


Y shape: (160, 40)
Y dtype: float32
Suma primera fila: 1.0
Mínimo: 2.3624361e-26 Máximo: 0.99999964
Guardado Y_pred.npz con clave 'Y' y shape (160, 40)


## Verificar archivo guardado

In [8]:
# Comprobación de que el archivo guardado se puede cargar correctamente
pred_file = np.load(OUT_PATH)
print("Claves del npz:", pred_file.files)
print("Shape guardada:", pred_file["Y"].shape)
print("Primeras 5 probabilidades de la primera imagen:", pred_file["Y"][0, :5])


Claves del npz: ['Y']
Shape guardada: (160, 40)
Primeras 5 probabilidades de la primera imagen: [4.7999688e-10 9.3216395e-09 9.5666008e-10 2.8897955e-14 4.9397437e-07]
